# 03 — Centralized Baseline Models
## Logistic Regression, Random Forest, XGBoost, XGB-RF Ensemble

This notebook trains centralized baselines on the BAF dataset (best balance of
interpretable features and size) and evaluates with 5-fold stratified CV.

Produces: V10 (baseline AUPRC comparison), V11 (CV box plot)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.models.baselines import train_baselines, get_baseline_models

FIGURES_DIR = Path('../outputs/figures')
MODELS_DIR = Path('../outputs/models')
TABLES_DIR = Path('../outputs/tables')
PROCESSED_DIR = Path('../data/processed')

for d in [FIGURES_DIR, MODELS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Load Preprocessed Data

In [ ]:
# Use BAF dataset as primary training dataset
# (interpretable features, 1M transactions, good fraud rate)
X_train = pd.read_csv(PROCESSED_DIR / 'baf_X_train.csv')
X_test = pd.read_csv(PROCESSED_DIR / 'baf_X_test.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'baf_y_train.csv').squeeze()
y_test = pd.read_csv(PROCESSED_DIR / 'baf_y_test.csv').squeeze()

print(f'Training set: {X_train.shape[0]:,} samples, {X_train.shape[1]} features')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'Train fraud rate: {y_train.mean()*100:.2f}%')
print(f'Test fraud rate:  {y_test.mean()*100:.2f}%')

## 2. Train All Baseline Models

In [ ]:
results = train_baselines(
    X_train, y_train,
    output_dir=str(MODELS_DIR)
)

# Summary
print('\n=== Cross-Validation Summary ===')
for name, res in results.items():
    print(f'{name:25s}: AUPRC = {res["cv_auprc_mean"]:.4f} +/- {res["cv_auprc_std"]:.4f}')

## 3. V10 — Baseline Model Comparison (AUPRC)

In [ ]:
model_names = list(results.keys())
means = [results[n]['cv_auprc_mean'] for n in model_names]
stds = [results[n]['cv_auprc_std'] for n in model_names]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(model_names, means, yerr=stds, capsize=5,
              color=colors[:len(model_names)], edgecolor='black', linewidth=0.5)

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{mean:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylabel('AUPRC (5-Fold CV)', fontsize=12)
ax.set_title('Centralized Baseline Model Comparison', fontsize=14)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_comparison_auprc.png', dpi=300)
plt.show()
print('Saved: baseline_comparison_auprc.png')

## 4. V11 — Cross-Validation Box Plot

In [ ]:
cv_data = [results[n]['cv_scores'] for n in model_names]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(cv_data, labels=model_names, patch_artist=True,
                widths=0.6, showmeans=True)

for patch, color in zip(bp['boxes'], colors[:len(model_names)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('AUPRC', fontsize=12)
ax.set_title('5-Fold Cross-Validation Performance', fontsize=14)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cv_boxplot_baselines.png', dpi=300)
plt.show()
print('Saved: cv_boxplot_baselines.png')

## 5. Test Set Evaluation

In [ ]:
from src.evaluation.metrics import full_evaluation

all_metrics = []
all_probas = {}

for name, res in results.items():
    model = res['model']
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = full_evaluation(y_test, y_pred, y_proba, name,
                              output_dir=str(FIGURES_DIR))
    all_metrics.append(metrics)
    all_probas[name] = y_proba
    print(f'{name}: AUPRC={metrics["AUPRC"]:.4f}, F1={metrics["F1-Score"]:.4f}, '
          f'Recall={metrics["Recall"]:.4f}')

# Save results
results_df = pd.DataFrame(all_metrics).set_index('Model').round(4)
results_df.to_csv(TABLES_DIR / 'baseline_test_results.csv')
print('\n')
print(results_df.to_string())